# nano-vLLM vs vLLM benchmark suite (Colab)

This notebook runs the full nano-vLLM benchmark catalog against an upstream vLLM server in a single Colab session, then aggregates the JSON results into one side-by-side summary.

**Model under test:** [`Qwen/Qwen3-0.6B`](https://huggingface.co/Qwen/Qwen3-0.6B). This is the only architecture currently registered in `nanovllm/models/qwen3.py` (`@ModelRegistry.register("qwen3", architectures=["Qwen3ForCausalLM"])`), and it is the default in `scripts/benchmark_kvcache_backends.py` and `scripts/benchmark_turboquant_compare.py`. The notebook downloads it directly into `/content/models/Qwen3-0.6B/` so every benchmark sees the same weights.

**What it does, in order:**
1. Clone (or update) the nano-vllm repo, install dependencies, and download `Qwen/Qwen3-0.6B`.
2. Run nano-vLLM offline benchmarks: `bench.py` and `scripts/benchmark_kvcache_backends.py` across multiple KV-cache backends.
3. Run the cross-engine comparison `scripts/benchmark_turboquant_compare.py` (nano-vLLM default vs TurboQuant vs vLLM auto vs vLLM TurboQuant).
4. Launch a vLLM OpenAI-compatible server in the background using `serve_vllm.sh`.
5. Drive that server with `scripts/benchmark_serving.py` (cache-hit + cold phases, streaming TTFT/ITL).
6. Print one combined summary table.

**Runtime requirements:** Colab T4 (free tier) or better. Switch the runtime to GPU via `Runtime → Change runtime type → GPU` before running.

**Note:** results below are written to `/content/benchmark_results/` so you can download them after the run.

## 0. Sanity check the runtime

In [2]:
!nvidia-smi || echo 'GPU not available; switch the Colab runtime to GPU and re-run.'
import sys
print('python:', sys.version)

Mon May 25 17:50:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 590.70                 Driver Version: 592.27         CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce GTX 1650        On  |   00000000:01:00.0 Off |                  N/A |
| N/A   57C    P8              5W /   50W |    1293MiB /   4096MiB |     27%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Clone the repo, install nano-vLLM, and fetch Qwen3-0.6B

Three steps:

1. `git clone` the nano-vllm repo (skipped if already present).
2. Run `setup_colab.sh` for the heavy dep install (PyTorch / Triton / Transformers / flash-attn / xxhash + editable nano-vllm install). The script ships with a model-download step that pulls `Qwen2.5-0.5B-Instruct` and renames the directory — that would not load against nano-vLLM (whose registry only knows `Qwen3ForCausalLM`), so we let setup_colab.sh exit however it likes (`|| true`) and then explicitly download the real `Qwen/Qwen3-0.6B`.
3. Download `Qwen/Qwen3-0.6B` into `/content/models/Qwen3-0.6B/` using `huggingface_hub.snapshot_download`, the same path the existing `scripts/benchmark_kvcache_backends.py::ensure_model` uses.

If you re-run this section after a successful first run, only the changed steps take work.

In [3]:
import os
REPO_URL = os.environ.get('NANOVLLM_REPO', 'https://github.com/GeeeekExplorer/nano-vllm.git')
REPO_DIR = '/content/nano-vllm'

if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    print(f'Repo already present at {REPO_DIR}; using existing checkout.')

%cd {REPO_DIR}
!ls -la

fatal: could not create leading directories of '/content/nano-vllm': Permission denied
[Errno 2] No such file or directory: '/content/nano-vllm'
/home/emmanuelalo52/nano-vllm/notebooks
total 28
drwxr-xr-x 2 emmanuelalo52 emmanuelalo52  4096 May 25 17:43 .
drwxr-xr-x 7 emmanuelalo52 emmanuelalo52  4096 May 25 17:43 ..
-rw-r--r-- 1 emmanuelalo52 emmanuelalo52 19650 May 25 17:50 benchmark_all_colab.ipynb


/home/emmanuelalo52/.local/lib/python3.10/site-packages/IPython/core/magics/osm.py:393: UserWarning: This is now an optional IPython functionality, using bookmarks requires you to install the `pickleshare` library.
  bkms = self.shell.db.get('bookmarks', {})


In [4]:
# 1a. Install deps via setup_colab.sh. The script's own model download targets
#     Qwen2.5-0.5B-Instruct (relabelled as Qwen3-0.6B) which nano-vLLM cannot
#     load, so we tolerate any non-zero exit code and re-download the right
#     model below.
!bash setup_colab.sh || echo 'setup_colab.sh exited non-zero; deps should still be installed. We re-download the real Qwen3 below.'

# 1b. Make sure huggingface_hub is available for snapshot_download.
!pip install -q --upgrade huggingface_hub

bash: setup_colab.sh: No such file or directory


In [ ]:
# 1c. Download the real Qwen/Qwen3-0.6B into /content/models/Qwen3-0.6B/.
#     This mirrors the ensure_model() helper in scripts/benchmark_kvcache_backends.py
#     so every downstream benchmark sees the same weights.
import os, shutil, pathlib
from huggingface_hub import snapshot_download

HF_MODEL_ID = os.environ.get('HF_MODEL_ID', 'Qwen/Qwen3-0.6B')
MODEL_DIR = '/content/models/Qwen3-0.6B'
pathlib.Path(MODEL_DIR).parent.mkdir(parents=True, exist_ok=True)

# If setup_colab.sh left a Qwen2.x download here, replace it cleanly so the
# directory only contains Qwen3 weights/config.
config_path = os.path.join(MODEL_DIR, 'config.json')
needs_download = True
if os.path.isfile(config_path):
    import json
    with open(config_path) as f:
        existing_arch = (json.load(f).get('architectures') or [None])[0]
    if existing_arch == 'Qwen3ForCausalLM':
        print(f'Found existing Qwen3 weights at {MODEL_DIR}; skipping download.')
        needs_download = False
    else:
        print(f'Found wrong architecture {existing_arch!r} at {MODEL_DIR}; replacing.')
        shutil.rmtree(MODEL_DIR)

if needs_download:
    print(f'Downloading {HF_MODEL_ID} -> {MODEL_DIR} ...')
    try:
        snapshot_download(
            repo_id=HF_MODEL_ID,
            local_dir=MODEL_DIR,
            local_dir_use_symlinks=False,
        )
    except TypeError:
        # Older huggingface_hub versions do not accept local_dir_use_symlinks.
        snapshot_download(repo_id=HF_MODEL_ID, local_dir=MODEL_DIR)
    print('Download complete.')

# Confirm we ended up with Qwen3.
import json as _json
with open(config_path) as f:
    arch = (_json.load(f).get('architectures') or [None])[0]
print('Final architecture:', arch)
assert arch == 'Qwen3ForCausalLM', f'Expected Qwen3ForCausalLM but got {arch!r}'

In [5]:
# Make sure the serving benchmark's HTTP client is installed.
!pip install -q aiohttp


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


## 2. Common configuration

Tune these knobs to fit your runtime. Defaults are conservative for the free T4.

In [6]:
import os, json, pathlib

HF_MODEL_ID = 'Qwen/Qwen3-0.6B'
MODEL_DIR = '/content/models/Qwen3-0.6B'
RESULTS_DIR = '/content/benchmark_results'
pathlib.Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)

# Workload knobs shared across benchmarks.
NUM_PROMPTS = 16
PREFIX_LEN = 512
SUFFIX_LEN = 64
MAX_TOKENS = 96
MAX_CONCURRENCY = 8

# KV-cache backends to exercise in the offline benchmark. All run against Qwen3-0.6B.
NANO_BACKENDS = 'default,turboquant_k4v4,turboquant_k3v4,saw_int4'

# vLLM serving knobs. SERVED_MODEL_NAME is just the logical alias the client uses;
# the weights are still Qwen3-0.6B loaded from MODEL_DIR.
SERVED_MODEL_NAME = 'qwen3-0.6b-benchmark'
SERVER_HOST = '127.0.0.1'
SERVER_PORT = '8000'

os.environ['FLASHINFER_DISABLE_VERSION_CHECK'] = '1'
os.environ['NANOVLLM_SAW_INT4_HADAMARD_ORDER'] = '16'

print(json.dumps({
    'HF_MODEL_ID': HF_MODEL_ID,
    'MODEL_DIR': MODEL_DIR,
    'RESULTS_DIR': RESULTS_DIR,
    'NUM_PROMPTS': NUM_PROMPTS,
    'PREFIX_LEN': PREFIX_LEN,
    'SUFFIX_LEN': SUFFIX_LEN,
    'MAX_TOKENS': MAX_TOKENS,
    'MAX_CONCURRENCY': MAX_CONCURRENCY,
    'NANO_BACKENDS': NANO_BACKENDS,
    'SERVED_MODEL_NAME': SERVED_MODEL_NAME,
}, indent=2))

PermissionError: [Errno 13] Permission denied: '/content'

## 3. Offline benchmark: `bench.py`

Same script the README references. Random prompts, single-pass throughput, no prefix-cache phase. Acts as a sanity check that the engine is healthy before the more elaborate suites run.

In [ ]:
# bench.py reads the model from ~/huggingface/Qwen3-0.6B by default; symlink
# the Colab path so we don't have to edit the file.
import os, pathlib
home_dir = pathlib.Path(os.path.expanduser('~/huggingface'))
home_dir.mkdir(parents=True, exist_ok=True)
link = home_dir / 'Qwen3-0.6B'
if not link.exists():
    link.symlink_to(MODEL_DIR)
    print(f'Linked {link} -> {MODEL_DIR}')
else:
    print(f'{link} already exists.')

!python3 bench.py 2>&1 | tee {RESULTS_DIR}/bench.log

## 4. Offline KV-cache backend sweep

`scripts/benchmark_kvcache_backends.py` runs each backend in a fresh subprocess so JIT state and CUDA memory do not leak. Reports cold + cache-hit phases for every backend plus its cache footprint.

In [ ]:
kv_json = f'{RESULTS_DIR}/kvcache_backends.json'
!python3 scripts/benchmark_kvcache_backends.py \
    --model {MODEL_DIR} \
    --backends {NANO_BACKENDS} \
    --num-prompts {NUM_PROMPTS} \
    --prefix-len {PREFIX_LEN} \
    --suffix-len {SUFFIX_LEN} \
    --max-tokens {MAX_TOKENS} \
    --max-model-len 2048 \
    --eager \
    --json-out {kv_json} 2>&1 | tee {RESULTS_DIR}/kvcache_backends.log

## 5. Cross-engine TurboQuant comparison

Runs nano-vLLM and vLLM side by side via `scripts/benchmark_turboquant_compare.py`. Each candidate runs in its own subprocess so CUDA state does not bleed across runs.

Both engines load `Qwen/Qwen3-0.6B`: nano-vLLM from the local directory at `MODEL_DIR`, and vLLM from the same path so weights match exactly (you can switch vLLM to the HF repo id instead by changing `--vllm-model Qwen/Qwen3-0.6B`).

On a T4 (SM75) vLLM TurboQuant is auto-skipped because its prefill path needs Ampere+; pass `--allow-unsupported-vllm-turboquant` if you are on a newer card and want to force it.

In [ ]:
tq_json = f'{RESULTS_DIR}/turboquant_compare.json'
!python3 scripts/benchmark_turboquant_compare.py \
    --nano-model {MODEL_DIR} \
    --vllm-model {MODEL_DIR} \
    --engines nanovllm,vllm \
    --nano-backends default,turboquant_k4v4 \
    --vllm-kv-cache-dtypes auto \
    --num-prompts {NUM_PROMPTS} \
    --prefix-len {PREFIX_LEN} \
    --suffix-len {SUFFIX_LEN} \
    --max-tokens {MAX_TOKENS} \
    --json-out {tq_json} 2>&1 | tee {RESULTS_DIR}/turboquant_compare.log

## 6. Launch the vLLM OpenAI-compatible server

`serve_vllm.sh` starts vLLM in the background, writes its PID to `vllm_server.pid` and the log to `vllm_server.log`. The script blocks until `/v1/models` answers (or `--no-wait` is passed).

In [ ]:
import os
os.environ['MODEL'] = MODEL_DIR
os.environ['SERVED_MODEL_NAME'] = SERVED_MODEL_NAME
os.environ['HOST'] = SERVER_HOST
os.environ['PORT'] = SERVER_PORT
os.environ['MAX_MODEL_LEN'] = '4096'
os.environ['MAX_NUM_SEQS'] = '128'
os.environ['GPU_MEMORY_UTILIZATION'] = '0.85'
os.environ['READY_TIMEOUT_S'] = '600'

!bash serve_vllm.sh

In [ ]:
# Quick sanity check that the server is talking and advertising our model.
import json, urllib.request
with urllib.request.urlopen(f'http://{SERVER_HOST}:{SERVER_PORT}/v1/models') as r:
    print(json.dumps(json.load(r), indent=2))

## 7. Online serving benchmark against vLLM

`scripts/benchmark_serving.py` reuses the same prompt-generation helpers as the offline benchmarks so the two workloads are comparable. It fires `cache_hit` and `cold` phases with bounded concurrency and reports request throughput, output throughput, TTFT and inter-token latency percentiles.

In [ ]:
serving_json = f'{RESULTS_DIR}/serving.json'
!python3 scripts/benchmark_serving.py \
    --base-url http://{SERVER_HOST}:{SERVER_PORT} \
    --model {SERVED_MODEL_NAME} \
    --tokenizer {MODEL_DIR} \
    --num-prompts {NUM_PROMPTS} \
    --max-concurrency {MAX_CONCURRENCY} \
    --prefix-len {PREFIX_LEN} \
    --suffix-len {SUFFIX_LEN} \
    --max-tokens {MAX_TOKENS} \
    --warmup-requests 4 \
    --stream \
    --json-out {serving_json} 2>&1 | tee {RESULTS_DIR}/serving.log

## 8. Stop the vLLM server

Frees the GPU before the final summary block runs (and keeps the runtime healthy if you re-execute upstream cells).

In [ ]:
import os, signal, time
pid_path = 'vllm_server.pid'
if os.path.exists(pid_path):
    pid = int(open(pid_path).read().strip())
    print(f'Stopping vLLM server PID={pid}')
    try:
        os.kill(pid, signal.SIGTERM)
        for _ in range(20):
            time.sleep(0.5)
            try:
                os.kill(pid, 0)
            except ProcessLookupError:
                break
        else:
            os.kill(pid, signal.SIGKILL)
    except ProcessLookupError:
        pass
    os.remove(pid_path)
    print('Stopped.')
else:
    print('No PID file; assuming the server is already stopped.')

## 9. Combined summary table

Reads every JSON we just wrote and prints one consolidated view: a row per (engine, config, phase) with wall time, throughput, latency, and (for the serving run) TTFT/ITL. This is the table to compare against the README's published numbers.

In [ ]:
import json, os

def safe_load(path):
    if not os.path.exists(path):
        return None
    try:
        with open(path) as f:
            return json.load(f)
    except json.JSONDecodeError as exc:
        print(f'WARN: could not parse {path}: {exc}')
        return None

rows = []  # (source, engine, config, phase, wall_s, out_tok_s, p50_latency_ms, p99_latency_ms)

# ---- 1. Offline KV-cache backend sweep ----
kv = safe_load(f'{RESULTS_DIR}/kvcache_backends.json') or []
for entry in kv:
    if not entry.get('ok'):
        rows.append(('offline_kvcache', 'nanovllm', entry.get('backend'), 'FAILED', None, None, None, None))
        continue
    for phase, metrics in (entry.get('phases') or {}).items():
        rows.append((
            'offline_kvcache',
            'nanovllm',
            entry.get('backend'),
            phase,
            metrics.get('wall_time_s'),
            metrics.get('end_to_end_tps'),
            None,
            None,
        ))

# ---- 2. Cross-engine TurboQuant comparison ----
tq = safe_load(f'{RESULTS_DIR}/turboquant_compare.json') or []
for entry in tq:
    if not entry.get('ok'):
        rows.append(('cross_engine', entry.get('engine'), entry.get('config'), 'FAILED', None, None, None, None))
        continue
    for phase, metrics in (entry.get('phases') or {}).items():
        rows.append((
            'cross_engine',
            entry.get('engine'),
            entry.get('config'),
            phase,
            metrics.get('wall_time_s'),
            metrics.get('output_tps'),
            None,
            None,
        ))

# ---- 3. Online serving benchmark ----
serving = safe_load(f'{RESULTS_DIR}/serving.json')
if serving:
    if not serving.get('ok'):
        rows.append(('online_serving', 'vllm', 'serve', 'FAILED', None, None, None, None))
    else:
        for phase, metrics in (serving.get('phases') or {}).items():
            rows.append((
                'online_serving',
                'vllm',
                'serve',
                phase,
                metrics.get('wall_time_s'),
                metrics.get('output_throughput'),
                metrics.get('latency_p50_ms'),
                metrics.get('latency_p99_ms'),
            ))

def fmt(value, digits=2):
    if value is None:
        return 'n/a'
    try:
        return f'{float(value):.{digits}f}'
    except (TypeError, ValueError):
        return 'n/a'

print(f'{"source":<16}{"engine":<10}{"config":<24}{"phase":<12}{"wall_s":>8}{"out_tok/s":>12}{"p50_lat_ms":>13}{"p99_lat_ms":>13}')
print('-' * 108)
for source, engine, config, phase, wall_s, out_tok_s, p50, p99 in rows:
    print(
        f'{source:<16}{(engine or ""):<10}{(config or ""):<24}{(phase or ""):<12}'
        f'{fmt(wall_s):>8}{fmt(out_tok_s, 1):>12}{fmt(p50, 1):>13}{fmt(p99, 1):>13}'
    )

print('\nRaw JSON files:')
for fname in os.listdir(RESULTS_DIR):
    print(' -', os.path.join(RESULTS_DIR, fname))

## 10. Download the results (optional)

Run this if you want the raw JSON locally for further analysis.

In [ ]:
import shutil, os
archive = '/content/benchmark_results.tar.gz'
shutil.make_archive(archive[:-7], 'gztar', RESULTS_DIR)
print('Wrote', archive, '({:.1f} KiB)'.format(os.path.getsize(archive) / 1024))
try:
    from google.colab import files
    files.download(archive)
except Exception as exc:
    print('Not running in Colab or download unavailable:', exc)